# Decision economics

A lender uses a model score to decide whether to approve each loan and to estimate what that
decision is worth. This requires values for both outcomes: the return from repayment and the
cost of default.

Because those outcomes are not symmetric and do not scale together, each loan has its own
break-even probability. This notebook applies that probability to the lending decision and
evaluates the result in currency rather than AUC.

What it shows:

- Compares three decision rules: approve every loan, use one break-even threshold for the book,
  and use a per-loan threshold that varies with the interest rate.
- Evaluates each rule using the cash that the loans repaid rather than the margin assumed by the
  model.
- Paired confidence intervals support the single threshold's higher profit within this
  validation period. The expected-profit rule's gain over approving every loan remains uncertain.
- Prices each loan using two values estimated on the training vintages: the margin earned per
  point of interest rate and the share of principal lost on default. The values are checked
  against the database before use.

In [1]:
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

from IPython.display import display
from scipy.stats import bootstrap

from credit_risk.data import (
    load_loans,
    load_outcomes,
    DB_PATH
)
from credit_risk.split import out_of_time_split
from credit_risk.model import (
    build_lgbm,
    LC_VERDICT_NUMERIC, LC_VERDICT_CATEGORICAL,
    UNDERWRITER_NUMERIC, UNDERWRITER_CATEGORICAL,
)
from credit_risk.evaluate import (
    expected_profit,
    breakeven_probability,
    MARGIN_PER_RATE_POINT,
    LOSS_FRACTION,
)

TARGET = "target_bad"
NUMERIC = UNDERWRITER_NUMERIC + LC_VERDICT_NUMERIC
CATEGORICAL = UNDERWRITER_CATEGORICAL + LC_VERDICT_CATEGORICAL

df = load_loans()
train, val, _ = out_of_time_split(df)

cols = NUMERIC + CATEGORICAL
pipe = build_lgbm(NUMERIC, CATEGORICAL)
pipe.fit(train[cols], train[TARGET])

proba = pipe.predict_proba(val[cols])
val = val.assign(proba=proba[:, 1])

print(
    f"validation {len(val)} loans, mean predicted PD {val['proba'].mean():.3f}, "
    f"bad rate {val[TARGET].mean():.3f}"
)

validation 154703 loans, mean predicted PD 0.130, bad rate 0.150


## What each outcome is worth

`evaluate.py` stores the margin slope and loss fraction as defaults, allowing the rule to run
without a database. `sql/30_loan_economics.sql` estimates both values on the training vintages
from payment columns that become available only after origination. They are used here to estimate
business parameters; using them as model features would cause leakage.

The stored defaults can diverge from the database estimates over time. The cell below reads the
estimates from the database and displays both sets of values together.

In [2]:
statements = Path("../sql/30_loan_economics.sql").read_text().split(";")
economics_sql = [s for s in statements if s.strip()][-1]

with duckdb.connect(str(DB_PATH), read_only=True) as con:
    estimated = con.execute(economics_sql).df().iloc[0]

pd.DataFrame({
    "from sql": {
        "margin_per_rate_point": estimated["margin_per_rate_point"],
        "loss_fraction": estimated["loss_fraction"],
    },
    "in evaluate.py": {
        "margin_per_rate_point": MARGIN_PER_RATE_POINT,
        "loss_fraction": LOSS_FRACTION,
    },
})

,from sql,in evaluate.py
margin_per_rate_point,0.0133,0.0133
loss_fraction,0.3543,0.3543


## Three policies

The notebook applies three rules to the same validation loans:

- **Approve all.** Fund every loan. This is the baseline against which the value of the other
  rules is measured.
- **Single break-even.** Approve a loan when its predicted default probability is below the
  break-even probability of the amount-weighted average rate. The threshold is derived from the
  economics rather than tuned on validation profit. The average rate is calculated on the
  training vintages, so the rule does not use the loans on which it is evaluated.
- **Expected profit.** Approve a loan when its expected profit is positive, which makes the
  threshold vary with the loan's rate.

Each rule is evaluated using each loan's realised cash flows rather than the margin slope used to
set the expected-profit threshold. The expected-profit rule is therefore evaluated against
observed cash flows rather than its own payoff assumptions.

In the table, `profit_per_loan` is the average profit per approved loan.

In [3]:
outcomes = load_outcomes().drop(columns=["loan_amnt"])
scored = val.merge(outcomes, on="id")
assert len(scored) == len(val)

scored["realised_profit"] = (
    scored["total_rec_prncp"] + scored["recoveries"] + scored["total_rec_int"] - scored["loan_amnt"]
)

scored["exp_profit"] = expected_profit(scored["proba"], scored["loan_amnt"], scored["int_rate"])

r_bar = (train["int_rate"] * train["loan_amnt"]).sum() / train["loan_amnt"].sum()
c = breakeven_probability(r_bar)
print(f"single break-even: {c:.3f}, from an average rate of {r_bar:.1f}%")

policies = {
    "approve all": pd.Series(True, index=scored.index),
    "single break-even": scored["proba"] < c,
    "expected profit": scored["exp_profit"] > 0,
}

rows = {}
for name, approve in policies.items():
    taken = scored[approve]
    rows[name] = {
        "approved": len(taken),
        "total_profit": taken["realised_profit"].sum(),
        "profit_per_loan": taken["realised_profit"].mean(),
        "bad_rate": taken[TARGET].mean(),
    }

policy_results = pd.DataFrame(rows).T
policy_results

single break-even: 0.316, from an average rate of 12.3%


,approved,total_profit,profit_per_loan,bad_rate
approve all,154703.0,1.321967e+08,854.519481,0.150016
single break-even,151111.0,1.331560e+08,881.179869,0.144278
expected profit,154302.0,1.323394e+08,857.665134,0.149382


### Uncertainty in the comparisons

We use a paired bootstrap because all three policies apply to the same validation loans. Under each policy, an approved loan contributes its realised profit and a rejected loan contributes zero. We calculate each loan's profit difference between two policies and resample these differences, keeping the comparison paired. The model, thresholds and approval decisions remain fixed.

Each table shows the difference measured on the full validation book and a 95% confidence interval, given by the 2.5th and 97.5th percentiles of 1,000 bootstrap samples with seed 0. Differences are calculated as first minus second, so positive values favour the first policy. Amounts are in millions.

The bootstrap includes all validation loans, including those rejected by either policy. The intervals describe sampling variability for comparable books of the same size; the cash totals already observed are known. They do not cover future-vintage drift, model refitting or uncertainty in the economic assumptions.

In [4]:
profits = pd.DataFrame({
    name: scored["realised_profit"].where(approve, 0)
    for name, approve in policies.items()
})
comparisons = {
    "single break-even - approve all": ("single break-even", "approve all"),
    "expected profit - approve all": ("expected profit", "approve all"),
    "single break-even - expected profit": ("single break-even", "expected profit"),
}

print(f"{len(scored):,} validation loans, 1,000 paired bootstrap samples")
for name, (first, second) in comparisons.items():
    difference = profits[first] - profits[second]
    interval = bootstrap(
        (difference,), np.sum, n_resamples=1000, batch=20, method="percentile", rng=0,
    ).confidence_interval

    print(name)
    display(pd.DataFrame({
        "difference": [difference.sum() / 1e6],
        "ci_low": [interval.low / 1e6],
        "ci_high": [interval.high / 1e6],
    }, index=["profit (M)"]).round(3))

154,703 validation loans, 1,000 paired bootstrap samples


single break-even - approve all


,difference,ci_low,ci_high
profit (M),0.959,0.294,1.628


expected profit - approve all


,difference,ci_low,ci_high
profit (M),0.143,-0.041,0.336


single break-even - expected profit


,difference,ci_low,ci_high
profit (M),0.817,0.188,1.461


The single break-even rule gains 0.959M over approving every loan, with a 95% interval of
[0.294, 1.628]M. Its gain over the expected-profit rule is 0.817M [0.188, 1.461]M. Both intervals
lie above zero, supporting the single threshold's advantage within this validation period.

The expected-profit rule gains 0.143M over approving every loan, but its interval of
[-0.041, 0.336]M spans zero, leaving that improvement uncertain. The observed differences remain
small relative to the 132.2M earned by approving every loan.

### Where the policies disagree

To understand why the single threshold performs better, we examine the loans approved by the
expected-profit rule but rejected by the single threshold. Their higher interest rates make
them attractive under the per-loan rule, despite their higher predicted default probabilities.

We compare three profit calculations for this group:

- **Predicted probabilities and estimated payoffs:** the profit expected when making the
  approval decision, using the model's PDs and the training-based margin and loss assumptions.
- **Observed defaults and estimated payoffs:** replace predicted probabilities with the actual
  repayment or default outcomes, while keeping the same margin and loss assumptions.
- **Realised cash flows:** calculate profit directly from the principal, interest and recoveries
  actually received, minus the amount lent.

The first comparison shows how the profit estimate changes when predicted default risk is
replaced by observed outcomes. The second checks whether the assumed margins and losses
adequately describe what these loans actually returned.

In [ ]:
be = scored["proba"] < c
ep = scored["exp_profit"] > 0

print(f"single break-even rejects {(~be).sum()}, expected profit rejects {(~ep).sum()}")

extra = scored[ep & ~be]
print(
    f"kept only by expected profit: {len(extra)} loans at "
    f"{extra['int_rate'].mean():.0f}% average rate, "
    f"{extra['proba'].mean():.0%} mean predicted PD, "
    f"{extra[TARGET].mean():.0%} observed default, "
    f"{extra['realised_profit'].sum() / 1e6:.1f}M realised"
)

profit_check = pd.Series({
    "training payoff at predicted probabilities": expected_profit(
        extra["proba"], extra["loan_amnt"], extra["int_rate"]
    ).sum(),
    "training payoff at observed outcomes": expected_profit(
        extra[TARGET], extra["loan_amnt"], extra["int_rate"]
    ).sum(),
    "realised cash flows": extra["realised_profit"].sum(),
}, name="total_profit")
profit_check

single break-even rejects 3592, expected profit rejects 401
kept only by expected profit: 3250 loans at 19% average rate, 35% mean predicted PD, 39% observed default, -0.8M realised


training payoff at predicted probabilities    1.670563e+06
training payoff at observed outcomes          2.149513e+05
realised cash flows                          -8.085393e+05
Name: total_profit, dtype: float64

The loans approved only by the expected-profit rule default more often than predicted: 39% versus a mean PD of 35%. Replacing predicted probabilities with observed outcomes reduces their estimated profit from 1.67M to 0.21M. Their realised cash flows show a loss of 0.81M, indicating that the margin and loss assumptions are also too favourable for this group.

The single threshold performs better largely because it rejects these loans. The expected-profit rule accepts their higher risk in exchange for higher interest rates, but the realised returns do not support that trade-off. Its weaker performance therefore reflects both underestimated default risk and inaccurate payoff assumptions, rather than calibration alone.

## Assumptions

The calculation does not discount future cash flows, so a euro received in month 36 is treated
like a euro received today. Prepayment is not modelled separately because the realised margin
already includes the lower interest earned when a loan is repaid early. The economic estimates
come from past loans and apply only while pricing and recovery behave as they did during that
period.

## Conclusions

The single break-even threshold produces the highest validation profit at 133.2M, compared with 132.3M for the expected-profit rule and 132.2M for approving every loan. Paired intervals support its advantage over both alternatives, while the expected-profit rule’s improvement over approving all remains uncertain. The overall difference is less than 1%, so policy choice has a modest financial effect within this portfolio.
The single threshold performs better mainly because it rejects high-rate loans that the expected-profit rule considers attractive. For these loans, the predicted default probability averages 35%, compared with 39% observed, and realised losses total 0.8M. The diagnostic checks point to both underestimated default risk and overly favourable margin and loss assumptions: higher interest rates did not compensate for risk as expected.

A more detailed decision rule therefore does not automatically produce better decisions. The per-loan approach depends on accurate probabilities and realistic payoffs, and errors in either can make risky loans appear profitable. In this validation period, the simpler threshold is less exposed to that problem.

These results concern only loans already funded by Lending Club and use undiscounted realised cash flows. They do not establish which policy would perform best across the full applicant pool or in future lending conditions.